# 04 · Detekcja mowy nienawiści – HerBERT-large (Polish Hate Speech)

Model: [`dkleczek/Polish-Hate-Speech-Detection-Herbert-Large`](https://huggingface.co/dkleczek/Polish-Hate-Speech-Detection-Herbert-Large) – HerBERT-large douczony na danych PolEval 2019 (mowa nienawiści).

Ten sam przebieg co w poprzednim projekcie (AMC_Hate_Speech, notebooki 04–07), na komentarzach, których używa zespół:

1. **Zbiór komentarzy** = filtr tematyczny zespołu (`scripts/topic_filter.py`): komentarz zostaje, gdy słownik obelg daje 1/1
   **albo** `scripts/topic_sample.is_potential_migration` znajduje temat migracji we wpisie lub komentarzu. Liczone w pamięci
   tym samym kodem – nic nie jest zapisywane do `topic/`.
2. **Najpierw słownik** (`src/lexicon/slurs.py`, ta sama reguła co `scripts/preprocess_slurs.py`, codebook 4.12):
   `hate_auto = 1` → hejt, model nie jest pytany.
3. **Model tylko dla pozostałych komentarzy.** Teksty dłuższe niż `MAX_LENGTH` tokenów są dzielone na nakładające się okna,
   `prob_hate_max` = maksimum po oknach.
4. `hate_final` = 1, gdy słownik **albo** model (`prob_hate_max ≥ THRESHOLD`); `label_source` = `slownik` / `{model}`.

**Wyjście:** `data/interim/<platforma>/<wydarzenie>/{OUT_DIRNAME}/comments.csv` (+ `meta.json`) oraz
`data/interim/<platforma>/{OUT_DIRNAME}_summary.csv`. Kolumny: `comment_id, in_topic, hate_auto, hate_migrants_auto, slurs,
slur_categories, slur_needs_context, prob_hate, prob_hate_max, hate_pred_max, n_tokens, n_windows, hate_final, label_source`.
Pliki nie zawierają treści komentarzy (tylko id i wyniki).

**Przerwanie:** wyniki zapisują się co `CHUNK_SIZE` tekstów (`_parts/`); po przerwaniu odpal notebook jeszcze raz.
Gotowe wydarzenia (zgodny `meta.json`) są pomijane, `FORCE = True` liczy od nowa.

In [ ]:
# ── KONFIGURACJA ──────────────────────────────────────────────────────────────
# PLATFORMS = ("wykop", "bluesky", "youtube")
# EVENTS = None                 # None = wszystkie wydarzenia z clean/; albo np. ["2024-06-06_Smierc_Zolnierza"]
PLATFORMS = ("wykop",)
EVENTS = ["2022-02-24_Atak_Rosji_Ukraina", "2022-05-08_Zabojstwo_Nowy_Swiat",
          "2023-09-22_Premiera_Zielona_Granica", "2023-10-15_Referendum_Wybory"]
MODEL_ID = "dkleczek/Polish-Hate-Speech-Detection-Herbert-Large"
OUT_DIRNAME = "herbert"
HATE_LABEL_INDEX = 1          # model nie ma id2label; 1 = hate (PolEval 2019)
THRESHOLD = 0.5               # prob_hate_max >= THRESHOLD -> hejt

MAX_LENGTH = 512              # tokeny na okno
TOKENS_PER_BATCH = 8192       # budżet tokenów na batch (mniej = mniej pamięci)
MAX_BATCH_SIZE = 128
WINDOW_OVERLAP = 128          # nakładka okien dla długich tekstów
CHUNK_SIZE = 2000             # co tyle tekstów zapis na dysk
USE_FP16 = True               # half precision na GPU/MPS (wyłącza się sam, gdy zmienia wyniki)
FORCE = False                 # True = policz od nowa

In [ ]:
# Jednorazowo, jeśli brakuje pakietów:
%pip install "torch>=2.6" transformers tqdm sacremoses

In [ ]:
import json, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "events.json").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.classifier.hate_models import Predictor, count_todo, event_dirs, run_event

def _read_env(path):
    env = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                env[k.strip()] = v.strip().strip('"').strip("'")
    return env

_env = _read_env(ROOT / ".env")
HF_TOKEN = _env.get("TOKEN") or _env.get("HF_TOKEN") or os.environ.get("HF_TOKEN")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
    torch.set_num_threads(os.cpu_count() or 4)

DIRS = event_dirs(PLATFORMS, EVENTS)
print(f"projekt : {ROOT}")
print(f"device  : {DEVICE} | torch {torch.__version__}")
print(f"wydarzenia: {len(DIRS)} ({', '.join(f'{p}: {sum(1 for k in DIRS if k[0] == p)}' for p in PLATFORMS)})")

PREP = None                   # tekst z clean/ bez zmian (jak w notebooku 04)
TOKEN = HF_TOKEN
SIGNATURE = {"model": MODEL_ID, "max_length": MAX_LENGTH, "hate_label_index": HATE_LABEL_INDEX, "topic": "topic_filter.py", "pipeline": "slownik->model v1"}

## 1. Zbiór komentarzy i ile pracy dla modelu

In [ ]:
# zbiór komentarzy (filtr tematyczny zespołu), ile rozstrzyga słownik, ile trafia do modelu
status = count_todo(DIRS, OUT_DIRNAME, SIGNATURE, force=FORCE)
display(status)
display(status.groupby("platform")[["clean", "topic", "slownik_1_1", "do_modelu"]].sum())
print(f"Do policzenia modelem: {status['do_modelu'].sum():,} komentarzy")

## 2. Model, sanity check i szacowany czas

In [ ]:
SANITY = [
    "Dzień dobry, jutro ma padać deszcz.",
    "Bardzo ciekawy artykuł, dzięki za wrzutkę.",
    "Uchodźcy powinni dostać pomoc, to ludzie jak my.",
    "Wszystkich tych brudasów trzeba wywalić z kraju, to zwierzęta.",
    "Ty pierdolony debilu, zdychaj.",
    "Imigranci to robactwo, które trzeba wytępić.",
]
t0 = time.time()
predictor = Predictor(MODEL_ID, DEVICE, prep=PREP, max_length=MAX_LENGTH, tokens_per_batch=TOKENS_PER_BATCH,
                      max_batch_size=MAX_BATCH_SIZE, window_overlap=WINDOW_OVERLAP, hate_label_index=HATE_LABEL_INDEX,
                      token=TOKEN, use_fp16=USE_FP16)
p, *_ = predictor.predict(SANITY)
print(f"model załadowany w {time.time() - t0:.1f}s | precyzja: {predictor.precision}")
display(pd.DataFrame({"tekst": SANITY, "prob_hate": p.round(3)}))
print("Ostatnie 3 powinny mieć WYSOKIE prob_hate, pierwsze 3 NISKIE – inaczej zmień HATE_LABEL_INDEX.")

# benchmark -> ETA
sample = []
for d in list(DIRS.values())[:8]:
    t = pd.read_csv(d / "clean" / "comments.csv", encoding="utf-8-sig", usecols=["text"], dtype=str,
                    keep_default_na=False)["text"]
    sample += t[t.str.strip() != ""].sample(min(64, len(t)), random_state=0).tolist()
predictor.predict(sample[:16])
t0 = time.time(); predictor.predict(sample)
rate = len(sample) / (time.time() - t0)
print(f"~{rate:.0f} tekstów/s -> szacunkowo {status['do_modelu'].sum() / rate / 60:.0f} min dla wszystkich platform")

## 3. Predykcje (można przerwać i wznowić)

In [ ]:
total = tqdm(total=int(status["do_modelu"].sum()), desc="WSZYSTKO", unit="txt", smoothing=0.05)
results = []
try:
    for (platform, ev), d in DIRS.items():
        r = run_event(predictor, d, OUT_DIRNAME, SIGNATURE, threshold=THRESHOLD, chunk_size=CHUNK_SIZE,
                      force=FORCE, total_bar=total, tqdm=tqdm)
        results.append(r)
        total.write(f"✔ {platform}/{ev}: {r['status']} | hate_final {r['hate_final']:,}/{r['topic']:,}")
except KeyboardInterrupt:
    total.write(f"⏸ Przerwano – policzone fragmenty są w */{OUT_DIRNAME}/_parts, odpal komórkę ponownie.")
finally:
    total.close()

## 4. Podsumowanie

In [ ]:
summary = pd.DataFrame(results)
if len(summary):
    cols = ["platform", "event", "clean", "topic", "slownik_1_1", "model_scored", "model_hate", "hate_final",
            "share_hate_final", "windowed", "lexicon_from"]
    summary = summary[cols]
    for platform, s in summary.groupby("platform"):
        p = ROOT / "data" / "interim" / platform / f"{OUT_DIRNAME}_summary.csv"
        s.to_csv(p, index=False, encoding="utf-8-sig")
        print("zapisano:", p.relative_to(ROOT))
    display(summary)
    display(summary.groupby("platform")[["topic", "slownik_1_1", "model_hate", "hate_final"]].sum()
            .assign(udzial_hate=lambda d: (d["hate_final"] / d["topic"]).round(4)))

In [ ]:
def load_with_text(platform, event_folder):
    d = ROOT / "data" / "interim" / platform / event_folder
    clean = pd.read_csv(d / "clean" / "comments.csv", encoding="utf-8-sig", dtype={"comment_id": str},
                        usecols=["comment_id", "relative_day", "text"])
    pred = pd.read_csv(d / OUT_DIRNAME / "comments.csv", encoding="utf-8-sig", dtype={"comment_id": str})
    return pred.merge(clean, on="comment_id", how="left")

pd.set_option("display.max_colwidth", 200)
(platform, ev) = next(iter(DIRS))
display(load_with_text(platform, ev).nlargest(10, "prob_hate_max")[["comment_id", "prob_hate_max", "label_source", "text"]])